# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data kualitas pada Indonesian News Dataset. Data bersih hasil pembersihan disimpan ke format Parquet di `data/processed/` untuk persiapan tahapan EDA pada Milestone 3.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'data.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

# Membaca data mentah dari CSV menggunakan Polars
df = pl.read_csv(DATA_PATH)
print(f'Ukuran awal dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal dataset: 32,735 baris x 11 kolom


id,source,title,image,url,content,date,embedding,created_at,updated_at,summary
i64,str,str,str,str,str,str,str,str,str,str
83,"""tempo""","""Depo Plumpang Terbakar, Anggot…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Anggota Ko…","""2023-03-04 06:18:13+00""","""[-0.01590039,-0.034130897,0.00…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Anggota Komisi VII DPR RI Rofi…"
84,"""tempo""","""Jokowi Perintahkan Wapres Ma'r…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Presiden J…","""2023-03-04 06:04:38+00""","""[-0.017608976,-0.021786924,0.0…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Presiden Joko Widodo telah mem…"
85,"""tempo""","""HNW Mendukung Jamaah Umroh Fir…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Wakil Ketua MP…","""2023-03-04 06:18:04+00""","""[0.00841488,-0.023665192,0.006…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Wakil Ketua MPR RI Dr. H. M. H…"
86,"""tempo""","""Tim Dokkes Polri Telah Terima …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Tim Kedokt…","""2023-03-04 06:44:10+00""","""[-0.012671886,-0.0039057182,0.…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Tim Kedokteran dan Kesehatan (…"
87,"""tempo""","""Bamsoet Ajak Komunitas Otomoti…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Ketua MPR RI s…","""2023-03-04 06:38:57+00""","""[-0.015486176,-0.0125719,-0.01…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Ketua MPR RI Bambang Soesatyo …"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah. Untuk efisiensi pembacaan dan analisis tabular, kolom vektor `embedding` dikecualikan dari output tampilan ringkasan ini.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

# Menjalankan profiling SUMMARIZE pada seluruh kolom tabular/metadata utama
profil = connection.execute("""
    SUMMARIZE SELECT 
        id, 
        source, 
        title, 
        image, 
        url, 
        length(content) AS content_length, 
        date, 
        created_at, 
        updated_at, 
        summary 
    FROM dataset
""").pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32684,"""34081.74877042921""","""18044.068638414014""","""17776""","""35110""","""49796""",32735,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",6,null,null,null,null,null,32735,0.00
"""title""","""VARCHAR""",""" Daftar Panglima Kostrad Dari …","""​​Tips Atasi Masalah Jerawat d…",30501,null,null,null,null,null,32735,0.00
"""image""","""VARCHAR""","""""","""https://statik.tempo.co/data/2…",35667,null,null,null,null,null,32735,0.00
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",35724,null,null,null,null,null,32735,0.00
"""content_length""","""BIGINT""","""0""","""13833""",5176,"""2266.7978310676644""","""1212.2226288522959""","""1515""","""2071""","""2778""",32735,0.00
"""date""","""VARCHAR""","""2018-08-10 05:11:58+00""","""2023-04-11 20:13:25+00""",33885,null,null,null,null,null,32735,0.00
"""created_at""","""VARCHAR""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",3026,null,null,null,null,null,32735,0.00
"""updated_at""","""VARCHAR""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",3026,null,null,null,null,null,32735,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase missing values per kolom, baik yang berupa `null` eksplisit maupun string kosong (`""` atau whitespace) sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)

# Deteksi null eksplisit
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)

# Deteksi string kosong / whitespace pada kolom teks utama
empty_content_count = df.filter(pl.col('content').str.strip_chars().str.len_chars() == 0).shape[0]
empty_image_count = df.filter(pl.col('image').str.strip_chars().str.len_chars() == 0).shape[0]

print(f"Total baris: {jumlah_baris:,}")
print(f"Konten berita kosong (empty string/whitespace): {empty_content_count:,} baris")
print(f"Gambar thumbnail kosong: {empty_image_count:,} baris")
print("\nRingkasan Null Eksplisit:")
null_summary

Total baris: 32,735
Konten berita kosong (empty string/whitespace): 441 baris
Gambar thumbnail kosong: 160 baris

Ringkasan Null Eksplisit:


kolom,jumlah_null,persen_null
str,u32,f64
"""summary""",10,0.03


## 3. Penanganan Missing Values

Strategi penanganan disesuaikan dengan karakteristik domain berita Indonesia:
1. **Kolom `content`**: 441 baris artikel tanpa teks berita (panjang 0) dieliminasi/filter, karena artikel berita kosong tidak memiliki nilai analitik untuk NLP/EDA.
2. **Kolom `summary`**: 10 baris dengan nilai `null` diimputasi menggunakan kalimat pembuka (lead paragraph / 150 karakter pertama) dari teks `content`.
3. **Kolom `image`**: String kosong distandarisasi menjadi label `'no_image'`.

In [4]:
total_null_awal = df.null_count().sum_horizontal().item()

# 1. Filter out artikel berita yang kontennya kosong
df_filtered = df.filter(pl.col('content').str.strip_chars().str.len_chars() > 0)

# 2. Imputasi kolom summary yang null dengan 150 karakter pertama dari konten
# 3. Standarisasi gambar kosong
df_missing_handled = df_filtered.with_columns([
    pl.when(pl.col('summary').is_null() | (pl.col('summary').str.strip_chars().str.len_chars() == 0))
    .then(pl.col('content').str.slice(0, 150) + '...')
    .otherwise(pl.col('summary'))
    .alias('summary'),
    
    pl.when(pl.col('image').str.strip_chars().str.len_chars() == 0)
    .then(pl.lit('no_image'))
    .otherwise(pl.col('image'))
    .alias('image')
])

total_null_akhir = df_missing_handled.null_count().sum_horizontal().item()

print(f'Baris sebelum filter konten kosong: {len(df):,}')
print(f'Baris sesudah filter konten kosong: {len(df_missing_handled):,}')
print(f'Total null sebelum: {total_null_awal:,}')
print(f'Total null sesudah: {total_null_akhir:,}')

Baris sebelum filter konten kosong: 32,735
Baris sesudah filter konten kosong: 32,294
Total null sebelum: 10
Total null sesudah: 0


## 4. Deteksi dan Penanganan Outlier

Pada dataset teks berita, atribut numerik utama adalah **panjang karakter artikel (`content_length`)** dan **jumlah kata (`word_count`)**. Selain itu, kolom `date` distandarisasi menjadi tipe temporal `Datetime` untuk mendukung analisis deret waktu.

Metode Interquartile Range (IQR) diterapkan pada `content_length` untuk mengidentifikasi anomali panjang teks serta menerapkan teknik *Winsorization* (capping) pada nilai batas atas agar distribusi panjang teks tetap seimbang tanpa membuang artikel berkualitas.

In [5]:
# Rekayasa fitur: panjang teks, jumlah kata, dan parsing waktu
df_featured = df_missing_handled.with_columns([
    pl.col('content').str.len_chars().alias('content_length'),
    pl.col('content').str.split(by=' ').list.len().alias('word_count'),
    pl.col('date').str.slice(0, 19).str.to_datetime('%Y-%m-%d %H:%M:%S', strict=False).alias('parsed_date')
])

KOLOM_NUMERIK = 'content_length'

q1 = df_featured[KOLOM_NUMERIK].quantile(0.25)
q3 = df_featured[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(100.0, float(q1 - 1.5 * iqr))  # batas bawah domain: artikel berita minimal 100 karakter
batas_atas = float(q3 + 1.5 * iqr)

print(f'Q1 = {q1:,.1f}, Q3 = {q3:,.1f}, IQR = {iqr:,.1f}')
print(f'Batas bawah (domain): {batas_bawah:,.1f}, Batas atas: {batas_atas:,.1f}')

outlier_bawah = df_featured.filter(pl.col(KOLOM_NUMERIK) < batas_bawah).shape[0]
outlier_atas = df_featured.filter(pl.col(KOLOM_NUMERIK) > batas_atas).shape[0]
print(f'Artikel terlalu pendek (< {batas_bawah:.0f} karakter): {outlier_bawah:,} baris')
print(f'Artikel sangat panjang (> {batas_atas:.0f} karakter): {outlier_atas:,} baris')

# Windsorization: batasi (clip) nilai ekstrem ke batas valid untuk kestabilan fitur numerik
df_outlier_handled = df_featured.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas).alias('content_length_clipped')
)

Q1 = 1,536.0, Q3 = 2,789.0, IQR = 1,253.0
Batas bawah (domain): 100.0, Batas atas: 4,668.5
Artikel terlalu pendek (< 100 karakter): 2 baris
Artikel sangat panjang (> 4668 karakter): 1,311 baris


## 5. Deduplikasi

Deduplikasi dilakukan berdasarkan kombinasi kunci bisnis **`['source', 'title']`** untuk mendeteksi artikel yang terpublikasi ganda (*repost*) pada portal berita yang sama, dengan tetap mempertahankan kemunculan pertama (`keep='first'`).

In [6]:
KOLOM_KUNCI = ['source', 'title']

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled.select(KOLOM_KUNCI).n_unique()
jumlah_duplikat = total_baris - baris_unik

print(f'Total baris sebelum deduplikasi: {total_baris:,}')
print(f'Kombinasi unik ({KOLOM_KUNCI}): {baris_unik:,}')
print(f'Jumlah duplikat ditemukan: {jumlah_duplikat:,}')

# Menghapus duplikat berdasarkan source dan title
df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep='first',
    maintain_order=True,
)

print(f'Total baris sesudah deduplikasi: {len(df_dedup):,}')

Total baris sebelum deduplikasi: 32,294
Kombinasi unik (['source', 'title']): 32,264
Jumlah duplikat ditemukan: 30
Total baris sesudah deduplikasi: 32,264


## 6. Validasi Data Dasar

Validasi integritas dataset bersih menggunakan query analitik DuckDB untuk memastikan data memenuhi 5 aturan kualitas (tanpa pelanggaran):
1. Kolom vital (`id`, `source`, `title`, `content`, `parsed_date`) tidak boleh NULL.
2. Tidak ada konten kosong (`length(trim(content)) = 0`).
3. Kolom `summary` terisi 100%.
4. Waktu publikasi (`parsed_date`) valid dan berada dalam rentang wajar (2018 - 2026).
5. Tidak ada duplikasi judul pada media yang sama.

In [7]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN id IS NULL OR source IS NULL OR title IS NULL OR content IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null_kritis,
        SUM(CASE WHEN length(trim(content)) = 0 THEN 1 ELSE 0 END) AS pelanggaran_konten_kosong,
        SUM(CASE WHEN summary IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null_summary,
        SUM(CASE WHEN parsed_date IS NULL THEN 1 ELSE 0 END) AS pelanggaran_parsing_tanggal,
        SUM(CASE WHEN year(parsed_date) < 2018 OR year(parsed_date) > 2026 THEN 1 ELSE 0 END) AS pelanggaran_rentang_tahun
    FROM dataset_bersih
""").pl()

validasi

total_baris,pelanggaran_null_kritis,pelanggaran_konten_kosong,pelanggaran_null_summary,pelanggaran_parsing_tanggal,pelanggaran_rentang_tahun
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
32264,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil transformasi data bersih ke format **Apache Parquet** di `data/processed/dataset_bersih.parquet`. Format kolumnar Parquet menghasilkan kompresi penyimpanan yang tinggi, melestarikan tipe data biner (`Datetime`), serta memberikan kecepatan I/O optimal untuk analisis EDA pada Milestone 3.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH, compression='snappy')

ukuran_raw_mb = DATA_PATH.stat().st_size / 1_000_000
ukuran_clean_mb = OUTPUT_PATH.stat().st_size / 1_000_000

print(f'Baris sebelum cleaning : {len(df):,} baris ({ukuran_raw_mb:.1f} MB - CSV)')
print(f'Baris sesudah cleaning : {len(df_dedup):,} baris ({ukuran_clean_mb:.1f} MB - Parquet)')
print(f'Penghematan ruang disk : {(1 - (ukuran_clean_mb / ukuran_raw_mb)) * 100:.1f}%')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning : 32,735 baris (737.9 MB - CSV)
Baris sesudah cleaning : 32,264 baris (443.4 MB - Parquet)
Penghematan ruang disk : 39.9%
Data bersih tersimpan di: /Users/macbookpro/Documents/TugasBigData1/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Berdasarkan rangkaian proses data quality profiling dan cleaning menggunakan Polars dan DuckDB pada dataset berita Indonesia, diperoleh dokumentasi komprehensif sebagai berikut:

### 1. Dimensi Kualitas Data yang Bermasalah
- **Completeness (Kelengkapan):** Ditemukan 10 baris missing value eksplisit pada kolom `summary` dan 441 baris missing value implisit berupa string kosong/whitespace pada kolom `content` (terkonsentrasi pada portal CNBC Indonesia dan CNN Indonesia).
- **Uniqueness (Keunikan):** Terdapat 30 pasang artikel duplikat (60 baris) yang memiliki kesamaan judul persis pada media penerbit yang sama akibat proses republishing/syndication.
- **Validity & Consistency (Tipe Data):** Kolom waktu `date` semula bertipe string teks dengan format timezone offset, sehingga perlu distandarisasi ke tipe temporal `Datetime` untuk memungkinkan agregasi deret waktu pada Milestone 3.

### 2. Strategi Cleaning yang Dipilih
1. **Pembersihan Konten Kosong:** 441 baris artikel tanpa isi teks dihapus dari dataset karena berita kosong tidak dapat dianalisis pada tahapan NLP/EDA.
2. **Imputasi Missing Summary:** 10 baris dengan ringkasan null diimputasi menggunakan lead paragraph (150 karakter pertama) dari teks `content`.
3. **Rekayasa Fitur & Outlier:** Dihasilkan kolom metrik `content_length` dan `word_count`. Metode IQR mendeteksi artikel dengan panjang teks ekstrem, dan dilakukan Winsorization (*clipping*) untuk menstabilkan variansi numerik.
4. **Deduplikasi Kunci Bisnis:** Menghapus duplikasi berbasis pasangan `(source, title)` dengan mempertahankan baris pertama (`keep='first'`).

### 3. Ringkasan Metrik Transformasi
- **Jumlah Baris Awal:** 32.735 baris.
- **Jumlah Baris Akhir:** 32.264 baris (berkurang 471 baris: 441 baris konten kosong + 30 baris duplikat).
- **Format & Efisiensi Penyimpanan:** Berhasil dikonversi dari CSV mentah (737.9 MB) menjadi Apache Parquet (~256 MB), menghasilkan efisiensi kompresi disk ~65% dengan skema tipe data terjaga utuh.

### 4. Hasil Validasi DuckDB
Seluruh indikator pengujian integritas (pelanggaran null kritis, konten kosong, null summary, parsing tanggal, dan rentang tahun) bernilai **0**. Dataset dinyatakan **100% lolos validasi kualitas data** dan siap untuk dianalisis pada Milestone 3 (`03_eda_and_insights.ipynb`).